# Workshop 02 — SFT-LoRA on Qwen3.5-4B VLM for Phishing Detection (Vision)

**Optional companion to notebook 01.** This trains a vision-language model on screenshots of phishing pages — the same task, different modality.

## Why this exists

Real phishing detection often relies on **how a page looks**, not just its text. A typosquatted PayPal page is obvious from the rendered visual even when the HTML is obfuscated.

Qwen3.5-4B is a vision-language model. Its JumpStart ID is `huggingface-vlm-qwen3-5-4b` and it supports SFT (LoRA).

## Prerequisite

Run `prep/02_prep_image_dataset.ipynb` first to publish a multimodal dataset to HuggingFace.

## Dataset format — verl multimodal SFT (`<image>` + top-level `images`)

SageMaker's managed SFT for VLMs runs on **verl** (`MultiTurnSFTDataset`), which has a specific multimodal schema — confirmed from verl source:

```json
{"messages": [
    {"role": "user",      "content": "<image>Is this page phishing or benign?"},
    {"role": "assistant", "content": "phish"}],
 "images": [{"bytes": <raw PNG bytes>}]}
```

Three rules that tripped us up (each is a hard requirement):

1. **Images go in a top-level `images` list**, *not* inside message content. Each entry is `{"bytes": <png>}` (or a PIL image). Inline `image_url`/`data:base64` parts in the content are **ignored** by verl.
2. **A literal `<image>` placeholder** in the message text marks where each image is spliced in. verl splits the string on `<image>` and consumes one image from the list per placeholder.
3. **No separate `system` turn.** verl tokenizes each turn in isolation, and the Qwen3.5 chat template raises *"System message must be at the beginning"* when a system turn is rendered alone. We fold the instruction into the user message instead.

Because images are raw bytes (not JSON-serializable as text), we write **Parquet**, not JSONL — the same as verl's own multimodal examples.

## 0. Install / upgrade SDK

In [ ]:
%pip install -q --upgrade sagemaker datasets boto3 rich pillow
# After this cell, **restart the kernel**.

## 1. Configuration

In [ ]:
HF_DATASET_ID = "gt2026workshop/phreshphish-2k"
HF_CONFIG_NAME = "image"
# S3 output — leave S3_BUCKET = None to use the account's DEFAULT SageMaker bucket
# (sagemaker-<region>-<account-id>). Session cell resolves S3_OUTPUT_PATH.
S3_BUCKET = None
S3_PREFIX = "qwen3-5-4b-phish-vlm-sft"
ROLE_ARN = None  # auto-detected

BASE_MODEL = "huggingface-vlm-qwen3-5-4b"   # Qwen3.5-4B VLM
MODEL_PACKAGE_GROUP_NAME = "qwen3-5-4b-phish-vlm-sft"
EXPERIMENT_NAME = "qwen3-5-4b-phish-vlm-sft"
RUN_NAME = "workshop-vlm-run-01"

ACCEPT_EULA = True

## 2. SageMaker session

In [ ]:
import os, boto3
from sagemaker.core.helper.session_helper import Session
from rich import print as rprint

REGION = boto3.Session().region_name
sm_client = boto3.client("sagemaker", region_name=REGION)
sagemaker_session = Session(sagemaker_client=sm_client)

# Default SageMaker bucket (sagemaker-<region>-<acct>); override via S3_BUCKET above.
BUCKET = S3_BUCKET or sagemaker_session.default_bucket()
S3_OUTPUT_PATH = f"s3://{BUCKET}/{S3_PREFIX}/"


if ROLE_ARN is None:
    from sagemaker.core.helper.session_helper import get_execution_role
    ROLE_ARN = get_execution_role()

os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{REGION}.app.aws"
rprint({"region": REGION, "role": ROLE_ARN, "output": S3_OUTPUT_PATH})

## 3. Pull the prepared image dataset and stage for SageMaker

The HF dataset has columns: `image` (PIL), `prompt` (str), `completion` (str).

We:
1. Downscale each image and build a verl record: `messages` (user `<image>...` + assistant label) plus a top-level `images=[{"bytes": ...}]`
2. Write one **Parquet** file per split (raw image bytes embedded)
3. Upload to S3 and register with the AI Registry

In [ ]:
import io, json, pathlib
import pandas as pd
from datasets import load_dataset
from urllib.parse import urlparse

ds = load_dataset(HF_DATASET_ID, name=HF_CONFIG_NAME)
print(ds)
print("Sample row keys:", list(ds["train"][0].keys()))

In [ ]:
p = urlparse(S3_OUTPUT_PATH.rstrip("/"))
BUCKET = p.netloc
PREFIX = p.path.lstrip("/")
DATA_PREFIX = f"{PREFIX}/data"

s3 = boto3.client("s3")
tmp = pathlib.Path("./_staged_vlm"); tmp.mkdir(exist_ok=True)

# Downscale so the longest side <= MAX_IMAGE_DIM (keeps Parquet + token count sane).
MAX_IMAGE_DIM = 768

# Instruction folded into the USER turn (verl has no standalone system turn).
INSTRUCTION = (
    "You are a phishing-detection classifier. Look at this screenshot of a webpage and "
    "answer with exactly one word: 'phish' if it is a phishing attempt, 'benign' if it is "
    "legitimate."
)

def image_to_png_bytes(pil_img, max_dim: int = MAX_IMAGE_DIM) -> bytes:
    """Resize (keep aspect) and return raw PNG bytes for the verl `images` list."""
    if max(pil_img.size) > max_dim:
        pil_img = pil_img.copy(); pil_img.thumbnail((max_dim, max_dim))
    buf = io.BytesIO()
    pil_img.convert("RGB").save(buf, format="PNG")
    return buf.getvalue()

def to_verl_record(image_pil, prompt: str, completion: str) -> dict:
    """verl MultiTurnSFTDataset multimodal record.

    - messages: user turn carries a single `<image>` placeholder + the instruction/prompt;
      assistant turn is the label. No system turn (verl renders turns in isolation and the
      Qwen3.5 template rejects a lone system message).
    - images: top-level list; one {"bytes": ...} per `<image>` placeholder.
    """
    user_text = f"<image>{INSTRUCTION}\n\n{prompt}"
    return {
        "messages": [
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": completion},
        ],
        "images": [{"bytes": image_to_png_bytes(image_pil)}],
    }

In [ ]:
from tqdm.auto import tqdm

def stage_split(split_name: str):
    records = [
        to_verl_record(r["image"], r["prompt"], r["completion"])
        for r in tqdm(ds[split_name], desc=f"build {split_name}")
    ]
    out_path = tmp / f"{split_name}.parquet"
    pd.DataFrame(records).to_parquet(out_path)
    size_mb = out_path.stat().st_size / 1e6
    key = f"{DATA_PREFIX}/{split_name}.parquet"
    s3.upload_file(str(out_path), BUCKET, key)
    s3_uri = f"s3://{BUCKET}/{key}"
    print(f"{split_name}: {len(records)} rows, {size_mb:.1f} MB \u2192 {s3_uri}")
    return s3_uri

TRAIN_S3 = stage_split("train")
VAL_S3 = stage_split("validation")

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

train_dataset = DataSet.create(name="phreshphish-workshop-image-train", source=TRAIN_S3, wait=True)
val_dataset = DataSet.create(name="phreshphish-workshop-image-val", source=VAL_S3, wait=True)
rprint({"train_arn": train_dataset.arn, "val_arn": val_dataset.arn})

## 4. Model package group + trainer

In [ ]:
from sagemaker.core.resources import ModelPackageGroup

try:
    mpg = ModelPackageGroup.create(
        model_package_group_name=MODEL_PACKAGE_GROUP_NAME,
        model_package_group_description="Qwen3.5-4B VLM fine-tuned for phishing screenshots",
    )
    print("created", mpg.model_package_group_arn)
except Exception:
    mpg = ModelPackageGroup.get(model_package_group_name=MODEL_PACKAGE_GROUP_NAME)
    print("reusing", MODEL_PACKAGE_GROUP_NAME)

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType
from rich.pretty import pprint

trainer = SFTTrainer(
    model=BASE_MODEL,
    training_type=TrainingType.LORA,
    model_package_group=mpg,
    training_dataset=train_dataset.arn,
    validation_dataset=val_dataset.arn,
    s3_output_path=S3_OUTPUT_PATH,
    sagemaker_session=sagemaker_session,
    accept_eula=ACCEPT_EULA,
    role=ROLE_ARN,
    mlflow_experiment_name=EXPERIMENT_NAME,
    mlflow_run_name=RUN_NAME,
)

print("Default hyperparameters for Qwen3.5-4B VLM + LoRA:")
pprint(trainer.hyperparameters.to_dict())

## 5. Train

In [ ]:
training_job = trainer.train(wait=True)
print("Job:", training_job.training_job_name, "|", training_job.training_job_status)

## 6. Deploy + smoke test

In [ ]:
import random
from sagemaker.serve import ModelBuilder

endpoint_name = f"qwen3-5-4b-vlm-phish-{random.randint(1000, 9999)}"
model_builder = ModelBuilder(model=trainer)
model_builder.build()
endpoint = model_builder.deploy(endpoint_name=endpoint_name)
print("Deployed:", endpoint_name)

In [ ]:
import json, base64, io, boto3
rt = boto3.client("sagemaker-runtime", region_name=REGION)

# NOTE: training data uses verl's <image> + top-level images format. The DEPLOYED
# endpoint serves with the OpenAI-style multimodal schema (image_url + data URL),
# which is the serving contract — different from the training contract. If the
# endpoint rejects this shape, check the served container's expected input schema.
def _img_b64(pil_img, max_dim=768):
    if max(pil_img.size) > max_dim:
        pil_img = pil_img.copy(); pil_img.thumbnail((max_dim, max_dim))
    buf = io.BytesIO(); pil_img.convert("RGB").save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()

def predict_vlm(image_pil, prompt: str) -> str:
    b64 = _img_b64(image_pil)
    # Instruction folded into the user turn, matching training (no system turn).
    user_text = (
        "You are a phishing-detection classifier. Look at this screenshot and answer with "
        "exactly one word: 'phish' or 'benign'.\n\n" + prompt
    )
    body = {
        "messages": [
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}"}},
                {"type": "text", "text": user_text},
            ]},
        ],
        "parameters": {"max_new_tokens": 5, "temperature": 0.0, "do_sample": False},
    }
    resp = rt.invoke_endpoint(EndpointName=endpoint_name, Body=json.dumps(body), ContentType="application/json")
    return json.loads(resp["Body"].read())

# Eval on first 10 held-out images
samples = ds["validation"].select(range(10))
for r in samples:
    out = predict_vlm(r["image"], r["prompt"])
    print(f"truth={r['completion']}  out={out}")

## 7. Clean up

```python
# sm_client.delete_endpoint(EndpointName=endpoint_name)
# sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)
```